# Deep Learning Based Skin Lesion Classification and Diagnosis

## CNN Training Notebook

This notebook implements the complete training pipeline for skin lesion classification using:
- **Dataset**: HAM10000 (10,015 dermoscopic images, 7 classes)
- **Model**: MobileNetV2 with Transfer Learning
- **Framework**: TensorFlow / Keras

### Table of Contents
1. Environment Setup & Imports
2. Dataset Loading & Exploration
3. Image Preprocessing
4. Data Augmentation
5. Model Architecture
6. Model Training (Two-Phase)
7. Evaluation & Metrics
8. Confusion Matrix
9. Grad-CAM Visualization
10. Save Model

---

## 1. Environment Setup & Imports

Install required dependencies and import all necessary libraries.

In [ ]:
# Install dependencies (uncomment if running on Google Colab)
# !pip install tensorflow opencv-python matplotlib seaborn scikit-learn pandas pillow

In [ ]:
# Core Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# Check GPU availability
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')
print(f'NumPy version: {np.__version__}')

## 2. Dataset Loading & Exploration

Load the HAM10000 metadata CSV and explore the class distribution.

**HAM10000 Dataset — 7 Classes (Table 5.1):**
| Class | Description | Count |
|-------|-------------|-------|
| nv | Melanocytic Nevi | 6705 |
| mel | Melanoma | 1113 |
| bkl | Benign Keratosis-like Lesions | 1099 |
| bcc | Basal Cell Carcinoma | 514 |
| akiec | Actinic Keratoses | 327 |
| vasc | Vascular Lesions | 142 |
| df | Dermatofibroma | 115 |

In [ ]:
# ============================================================
# CONFIGURATION — Update these paths for your environment
# ============================================================
DATASET_DIR = '../dataset'  # Path to dataset folder
METADATA_FILE = os.path.join(DATASET_DIR, 'HAM10000_metadata.csv')
IMAGES_DIR = os.path.join(DATASET_DIR, 'HAM10000_images')
MODEL_SAVE_PATH = '../models/skin_lesion_model.h5'

# For Google Colab, update paths:
# DATASET_DIR = '/content/drive/MyDrive/skin-lesion-classifier/dataset'

# Image parameters
IMG_SIZE = (224, 224)
IMG_SHAPE = (224, 224, 3)
BATCH_SIZE = 16
NUM_CLASSES = 7

# Class mappings
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_FULL = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions'
}

print('Configuration loaded successfully')

In [ ]:
# Load metadata CSV
df = pd.read_csv(METADATA_FILE)
print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
df.head(10)

In [ ]:
# Display class distribution
print('HAM10000 Dataset — Class Distribution (Table 5.1):')
print('=' * 45)
class_dist = df['dx'].value_counts()
for cls, count in class_dist.items():
    print(f'  {cls:8s} | {CLASS_FULL[cls]:30s} | {count:5d} images')
print(f'{"":8s} | {"TOTAL":30s} | {len(df):5d} images')

In [ ]:
# Plot class distribution
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E91E63', '#9C27B0', '#2196F3', '#00BCD4', '#FF5722', '#4CAF50', '#FF9800']
bars = ax.bar(class_dist.index, class_dist.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, class_dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('HAM10000 Dataset — Class Distribution', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Skin Lesion Class', fontsize=13)
ax.set_ylabel('Number of Images', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../static/images/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, cls in enumerate(CLASS_NAMES):
    sample = df[df['dx'] == cls].iloc[0]
    img_path = os.path.join(IMAGES_DIR, sample['image_id'] + '.jpg')
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(img)
    axes[idx].set_title(f"{cls}\n{CLASS_FULL[cls]}", fontsize=10, fontweight='bold')
    axes[idx].axis('off')

# Hide the extra subplot
axes[7].axis('off')
plt.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Image Preprocessing

Implement preprocessing steps from **Table 5.2**:
1. **Image Resizing**: 224×224 pixels
2. **Hair Removal (DullRazor)**: Morphological blackhat + inpainting
3. **Contrast Enhancement (CLAHE)**: Adaptive histogram equalization
4. **Normalization**: Scale to [0, 1] range

In [ ]:
def remove_hair(image):
    """Remove hair artifacts using DullRazor algorithm (Table 5.2)"""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    blackhat_blurred = cv2.GaussianBlur(blackhat, (3, 3), 0)
    _, hair_mask = cv2.threshold(blackhat_blurred, 10, 255, cv2.THRESH_BINARY)
    clean_image = cv2.inpaint(image, hair_mask, inpaintRadius=6, flags=cv2.INPAINT_TELEA)
    return clean_image


def enhance_contrast(image):
    """Enhance contrast using CLAHE (Table 5.2)"""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return enhanced


def preprocess_image(image_path):
    """Complete preprocessing pipeline"""
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.resize(img, IMG_SIZE)
    img = remove_hair(img)
    img = enhance_contrast(img)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    return img


print('Preprocessing functions defined successfully')

In [ ]:
# Demonstrate preprocessing on a sample image
sample = df.iloc[0]
sample_path = os.path.join(IMAGES_DIR, sample['image_id'] + '.jpg')
original = cv2.imread(sample_path)
original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

# Apply each step
resized = cv2.resize(original, IMG_SIZE)
hair_removed = remove_hair(resized)
contrast_enhanced = enhance_contrast(hair_removed)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original', 'Resized (224×224)', 'Hair Removed', 'Contrast Enhanced']
images = [original_rgb, cv2.cvtColor(resized, cv2.COLOR_BGR2RGB),
          cv2.cvtColor(hair_removed, cv2.COLOR_BGR2RGB),
          cv2.cvtColor(contrast_enhanced, cv2.COLOR_BGR2RGB)]
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('Image Preprocessing Pipeline (Table 5.2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Preparation & Augmentation

- Handle class imbalance via oversampling
- Apply data augmentation (Table 5.3)
- Split: 80% Train / 20% Test (Chapter 6.2)

In [ ]:
# Add file path column
df['filepath'] = df['image_id'].apply(
    lambda x: os.path.join(IMAGES_DIR, x + '.jpg')
)

# Verify images exist
df['exists'] = df['filepath'].apply(os.path.exists)
print(f"Images found: {df['exists'].sum()} / {len(df)}")
df = df[df['exists']].drop(columns=['exists'])

# Map labels to indices
df['label'] = df['dx'].map({name: idx for idx, name in enumerate(CLASS_NAMES)})
print(f'\nDataset ready: {len(df)} images')

In [ ]:
# Handle class imbalance via oversampling
target_count = int(df['dx'].value_counts().mean())
print(f'Target samples per class: {target_count}')

balanced_dfs = []
for cls in CLASS_NAMES:
    cls_df = df[df['dx'] == cls]
    if len(cls_df) < target_count:
        additional = target_count - len(cls_df)
        oversampled = cls_df.sample(n=additional, replace=True, random_state=42)
        cls_df = pd.concat([cls_df, oversampled], ignore_index=True)
        print(f'  {cls}: oversampled to {len(cls_df)}')
    else:
        print(f'  {cls}: kept {len(cls_df)}')
    balanced_dfs.append(cls_df)

df_balanced = pd.concat(balanced_dfs, ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\nBalanced dataset: {len(df_balanced)} images')

In [ ]:
# Split dataset: 80% Train, 20% Test (Chapter 6.2)
train_df, test_df = train_test_split(
    df_balanced, test_size=0.20, stratify=df_balanced['label'], random_state=42
)

# Further split: 15% of train for validation
train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df['label'], random_state=42
)

print(f'Training:   {len(train_df)} images')
print(f'Validation: {len(val_df)} images')
print(f'Testing:    {len(test_df)} images')

In [ ]:
# Create data generators with augmentation (Table 5.3)
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    shear_range=0.15
)

val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create flows from DataFrames
train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='dx', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, shuffle=True
)

val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='dx', target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, shuffle=False
)

test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='dx', target_size=IMG_SIZE,
    batch_size=1, class_mode='categorical', classes=CLASS_NAMES, shuffle=False
)

print(f'\nGenerators created successfully')

## 5. Model Architecture

Build the CNN model using **MobileNetV2 Transfer Learning** (Table 5.4):

```
Input (224×224×3)
  → MobileNetV2 (ImageNet, frozen)
  → GlobalAveragePooling2D
  → Dense(256, ReLU) → BatchNorm → Dropout(0.5)
  → Dense(128, ReLU) → Dropout(0.3)
  → Dense(7, Softmax)
```

In [ ]:
# Build model with MobileNetV2 transfer learning
base_model = MobileNetV2(input_shape=IMG_SHAPE, include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze base layers for Phase 1

# Custom classification head
inputs = Input(shape=IMG_SHAPE)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D(name='global_avg_pooling')(x)
x = Dense(256, activation='relu', name='dense_256')(x)
x = BatchNormalization(name='batch_norm')(x)
x = Dropout(0.5, name='dropout_50')(x)
x = Dense(128, activation='relu', name='dense_128')(x)
x = Dropout(0.3, name='dropout_30')(x)
outputs = Dense(NUM_CLASSES, activation='softmax', name='classification')(x)

model = Model(inputs=inputs, outputs=outputs, name='SkinLesionClassifier')

print(f'Base model layers: {len(base_model.layers)} (FROZEN)')
model.summary()

In [ ]:
# Compute class weights for handling imbalance
class_weights = compute_class_weight(
    'balanced', classes=np.unique(train_df['label']), y=train_df['label']
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:')
for idx, w in class_weight_dict.items():
    print(f'  {CLASS_NAMES[idx]}: {w:.4f}')

## 6. Model Training

**Two-Phase Training Strategy:**

- **Phase 1** (Feature Extraction): Frozen base, LR=1e-3, 5 epochs
- **Phase 2** (Fine-Tuning): Unfreeze top 50 layers, LR=1e-5, 15 epochs

Training parameters (Chapter 6.2):
- Optimizer: Adam
- Loss: Categorical Cross-Entropy
- Batch Size: 16

In [ ]:
# Callbacks
os.makedirs('../models', exist_ok=True)
callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print('Callbacks configured')

In [ ]:
# ============================================================
# PHASE 1: Feature Extraction (Frozen Base)
# ============================================================
print('=' * 60)
print('PHASE 1: Feature Extraction (Base FROZEN)')
print('  Epochs: 5, Learning Rate: 1e-3')
print('=' * 60)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_phase1 = model.fit(
    train_gen,
    epochs=5,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============================================================
# PHASE 2: Fine-Tuning (Unfreeze Top Layers)
# ============================================================
print('\n' + '=' * 60)
print('PHASE 2: Fine-Tuning (Top 50 layers UNFROZEN)')
print('  Epochs: 15, Learning Rate: 1e-5')
print('=' * 60)

# Unfreeze top 50 layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Save final model
model.save(MODEL_SAVE_PATH)
print(f'\nModel saved to: {MODEL_SAVE_PATH}')

## 7. Evaluation & Metrics

Evaluate model performance (Chapter 6.3):
- Accuracy
- Precision
- Recall
- F1-Score
- Training vs Validation graphs

In [ ]:
# Combine training histories
history = {}
for key in history_phase1.history:
    history[key] = history_phase1.history[key] + history_phase2.history[key]

epochs = range(1, len(history['accuracy']) + 1)

In [ ]:
# Figure 6.2: Training and Validation Accuracy Graph
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, history['accuracy'], 'b-o', label='Training Accuracy', linewidth=2, markersize=4)
ax.plot(epochs, history['val_accuracy'], 'r-o', label='Validation Accuracy', linewidth=2, markersize=4)
ax.axvline(x=5, color='green', linestyle='--', alpha=0.7, label='Fine-tuning starts')
ax.set_title('Training and Validation Accuracy', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])
plt.tight_layout()
os.makedirs('../static/images', exist_ok=True)
plt.savefig('../static/images/training_validation_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6.3: Training and Validation Loss Graph
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs, history['loss'], 'b-o', label='Training Loss', linewidth=2, markersize=4)
ax.plot(epochs, history['val_loss'], 'r-o', label='Validation Loss', linewidth=2, markersize=4)
ax.axvline(x=5, color='green', linestyle='--', alpha=0.7, label='Fine-tuning starts')
ax.set_title('Training and Validation Loss', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../static/images/training_validation_loss.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate on test set
print('Evaluating model on test set...')
test_gen.reset()
predictions = model.predict(test_gen, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_gen.classes
y_pred = y_pred[:len(y_true)]

# Overall metrics
from sklearn.metrics import precision_score, recall_score, f1_score
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f'\n{"="*50}')
print(f'  EVALUATION RESULTS (Chapter 6.3)')
print(f'{"="*50}')
print(f'  Accuracy:   {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'  Precision:  {precision:.4f} ({precision*100:.1f}%)')
print(f'  Recall:     {recall:.4f} ({recall*100:.1f}%)')
print(f'  F1-Score:   {f1:.4f} ({f1*100:.1f}%)')
print(f'{"="*50}')

In [ ]:
# Classification Report (per-class)
print('\nPer-Class Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Figure 6.4: Model Performance Evaluation Metrics
metrics = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(metrics.keys(), [v*100 for v in metrics.values()],
              color=['#2196F3', '#4CAF50', '#FF9800', '#E91E63'],
              edgecolor='white', linewidth=1.5, width=0.6)
for bar, v in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v*100:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_title('Model Performance Evaluation Metrics', fontsize=16, fontweight='bold', pad=15)
ax.set_ylabel('Score (%)', fontsize=13)
ax.set_ylim([0, 105])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../static/images/performance_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Confusion Matrix

**Figure 6.5**: Confusion Matrix of Skin Lesion Classification Model

In [ ]:
# Figure 6.5: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_title('Confusion Matrix — Skin Lesion Classification', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
plt.tight_layout()
plt.savefig('../static/images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Grad-CAM Visualization

Generate Grad-CAM heatmaps to visually explain model predictions.
Highlights the image regions most influential in classification.

In [ ]:
def get_gradcam_heatmap(model, image, pred_index=None):
    """Generate Grad-CAM heatmap for a prediction"""
    # Find last conv layer
    last_conv = None
    for layer in model.layers:
        if hasattr(layer, 'layers'):
            for sub_layer in layer.layers:
                if isinstance(sub_layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                    last_conv = sub_layer
        elif isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
            last_conv = layer
    
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[last_conv.output, model.output]
    )
    
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_score = predictions[:, pred_index]
    
    gradients = tape.gradient(class_score, conv_outputs)
    pooled = tf.reduce_mean(gradients, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

print('Grad-CAM function defined')

In [ ]:
# Demonstrate Grad-CAM on sample images
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for idx in range(3):
    # Get a sample image
    sample = test_df.iloc[idx * 50]
    img_path = sample['filepath']
    true_label = CLASS_NAMES[sample['label']]
    
    # Load and preprocess
    img = cv2.imread(img_path)
    img = cv2.resize(img, IMG_SIZE)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_input = img_rgb.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_input, axis=0)
    
    # Predict
    pred = model.predict(img_batch, verbose=0)
    pred_class = CLASS_NAMES[np.argmax(pred[0])]
    confidence = np.max(pred[0]) * 100
    
    # Generate Grad-CAM
    heatmap = get_gradcam_heatmap(model, img_batch)
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(img_rgb, 0.6, heatmap_colored, 0.4, 0)
    
    # Plot
    axes[idx, 0].imshow(img_rgb)
    axes[idx, 0].set_title(f'Original\nTrue: {true_label}', fontsize=11)
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(heatmap_resized, cmap='jet')
    axes[idx, 1].set_title('Grad-CAM Heatmap', fontsize=11)
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(overlay)
    axes[idx, 2].set_title(f'Overlay\nPred: {pred_class} ({confidence:.1f}%)', fontsize=11)
    axes[idx, 2].axis('off')

plt.suptitle('Grad-CAM Visualizations — Explainable AI', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/images/gradcam_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

### Training Complete!

**Model**: MobileNetV2 with Transfer Learning

**Results**:
- The model successfully classifies dermoscopic skin lesion images into 7 categories
- Two-phase training with feature extraction and fine-tuning
- Grad-CAM provides visual explanations for predictions

**Saved Files**:
- Model: `models/skin_lesion_model.h5`
- Evaluation plots: `static/images/`

**Next Steps**:
1. Run the Flask web app: `python app.py`
2. Upload test images at `http://localhost:5000`
3. View predictions with Grad-CAM explanations